# 01 — Ingestão, QC e Manifest

**Objetivo:** Descobrir a estrutura dos CSVs do RAVDESS (Kaggle landmarks),
extrair metadados (ator, emoção), gerar `manifest.csv` e `qc_report.csv`
com estatísticas e visualizações.

**Outputs:**
- `data/ravdess_landmarks_kaggle/03_qc/manifest.csv`
- `data/ravdess_landmarks_kaggle/03_qc/qc_report.csv`
- Visualizações inline

In [ ]:
import sys, os

# Garantir que src/ esteja no path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.ravdess_utils import (
    find_all_csvs,
    discover_csv_structure,
    parse_ravdess_filename,
    build_manifest,
    EMOTION_MAP,
    EMOTION_LABELS,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("Imports OK")

## 1.1 Configuração de Paths

In [ ]:
# Paths principais
RAW_DIR = os.path.join(ROOT, 'data', 'ravdess_landmarks_kaggle', '00_raw_kaggle_csv')
QC_DIR  = os.path.join(ROOT, 'data', 'ravdess_landmarks_kaggle', '03_qc')

os.makedirs(QC_DIR, exist_ok=True)

print(f"Diretório de dados brutos: {RAW_DIR}")
print(f"Diretório de QC:           {QC_DIR}")
print(f"Existe RAW_DIR? {os.path.isdir(RAW_DIR)}")

## 1.2 Descoberta dos CSVs

In [ ]:
csv_files = find_all_csvs(RAW_DIR)
print(f"Total de CSVs encontrados: {len(csv_files)}")

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"Nenhum CSV encontrado em {RAW_DIR}.\n"
        "Coloque os CSVs do Kaggle neste diretório e re-execute."
    )

# Mostrar primeiros arquivos
for f in csv_files[:5]:
    print(f"  {os.path.basename(f)}")
print(f"  ... ({len(csv_files)} total)")

## 1.3 Descoberta automática da estrutura dos CSVs

Vamos inspecionar a estrutura de alguns CSVs para entender as colunas disponíveis.

In [ ]:
# Inspecionar estrutura dos primeiros 3 CSVs
print("=" * 70)
print("INSPEÇÃO DA ESTRUTURA DOS CSVs")
print("=" * 70)

for csv_path in csv_files[:3]:
    info = discover_csv_structure(csv_path)
    print(f"\nArquivo: {os.path.basename(csv_path)}")
    print(f"  Colunas ({info['n_cols']}): {info['columns'][:10]}..." if info['n_cols'] > 10 else f"  Colunas ({info['n_cols']}): {info['columns']}")
    print(f"  Shape amostra: {info['shape_sample']}")

In [ ]:
# Mostrar DataFrame completo do primeiro CSV para entender layout
sample_df = pd.read_csv(csv_files[0])
print(f"Shape do primeiro CSV: {sample_df.shape}")
print(f"\nColunas ({len(sample_df.columns)}):")
for i, col in enumerate(sample_df.columns):
    print(f"  [{i:3d}] {col} ({sample_df[col].dtype})")

print(f"\nPrimeiras 3 linhas:")
sample_df.head(3)

## 1.4 Parsing de filename RAVDESS

O padrão RAVDESS codifica metadados no nome do arquivo:
- `03-01-{emotion}-{intensity}-{statement}-{rep}-{actor}.csv`

Mapeamento de emoções (8 classes):
1. Neutral, 2. Calm, 3. Happy, 4. Sad, 5. Angry, 6. Fearful, 7. Disgust, 8. Surprised

In [ ]:
# Testar parsing com alguns arquivos
print("Teste de parsing RAVDESS:")
print("-" * 60)
for f in csv_files[:5]:
    meta = parse_ravdess_filename(f)
    print(f"  {os.path.basename(f)}")
    if meta:
        print(f"    -> emoção={meta['emotion_label']} (code={meta['emotion_code']}), ator={meta['actor_id']:02d}")
    else:
        print(f"    -> PARSE FALHOU")

## 1.5 Construção do Manifest

Vamos iterar por todos os CSVs, extrair metadados e gerar o manifest completo.

In [ ]:
%%time
# Construir manifest completo
manifest = build_manifest(csv_files)

print(f"\nManifest gerado: {manifest.shape}")
print(f"Amostras com status_ok=True:  {manifest['status_ok'].sum()}")
print(f"Amostras com status_ok=False: {(~manifest['status_ok']).sum()}")
print(f"Amostras com NaN: {manifest['has_nan'].sum()}")

manifest.head(10)

In [ ]:
# Salvar manifest
manifest_path = os.path.join(QC_DIR, 'manifest.csv')
manifest.to_csv(manifest_path, index=False)
print(f"Manifest salvo em: {manifest_path}")

## 1.6 Estatísticas e QC Report

In [ ]:
# Filtrar apenas amostras válidas para estatísticas
valid = manifest[manifest['status_ok'] == True].copy()
print(f"Amostras válidas: {len(valid)} / {len(manifest)}")

# Estatísticas básicas
print(f"\n--- Estatísticas de Frames ---")
print(valid['n_frames'].describe())

print(f"\n--- Distribuição por Emoção ---")
emotion_counts = valid['emotion_label'].value_counts().sort_index()
print(emotion_counts)

print(f"\n--- Distribuição por Ator ---")
actor_counts = valid['actor_id'].value_counts().sort_index()
print(actor_counts)

In [ ]:
# Gerar QC report
qc_records = []

# Resumo geral
qc_records.append({
    'metric': 'total_csvs',
    'value': len(manifest),
})
qc_records.append({
    'metric': 'valid_csvs',
    'value': len(valid),
})
qc_records.append({
    'metric': 'invalid_csvs',
    'value': len(manifest) - len(valid),
})
qc_records.append({
    'metric': 'csvs_with_nan',
    'value': int(manifest['has_nan'].sum()),
})
qc_records.append({
    'metric': 'n_actors',
    'value': valid['actor_id'].nunique(),
})
qc_records.append({
    'metric': 'n_emotions',
    'value': valid['emotion_code'].nunique(),
})
qc_records.append({
    'metric': 'n_features',
    'value': int(valid['n_features'].mode().iloc[0]) if len(valid) > 0 else 0,
})
qc_records.append({
    'metric': 'frames_min',
    'value': int(valid['n_frames'].min()) if len(valid) > 0 else 0,
})
qc_records.append({
    'metric': 'frames_max',
    'value': int(valid['n_frames'].max()) if len(valid) > 0 else 0,
})
qc_records.append({
    'metric': 'frames_mean',
    'value': round(valid['n_frames'].mean(), 1) if len(valid) > 0 else 0,
})
qc_records.append({
    'metric': 'frames_median',
    'value': round(valid['n_frames'].median(), 1) if len(valid) > 0 else 0,
})

# Distribuição por emoção
for emo_code, emo_label in sorted(EMOTION_MAP.items()):
    count = len(valid[valid['emotion_code'] == emo_code])
    qc_records.append({
        'metric': f'emotion_{emo_code}_{emo_label}',
        'value': count,
    })

qc_df = pd.DataFrame(qc_records)
qc_path = os.path.join(QC_DIR, 'qc_report.csv')
qc_df.to_csv(qc_path, index=False)
print(f"QC report salvo em: {qc_path}")
qc_df

## 1.7 Visualizações

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Histograma de frames
axes[0].hist(valid['n_frames'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Número de Frames')
axes[0].set_ylabel('Contagem')
axes[0].set_title('Distribuição do Número de Frames por Amostra')
axes[0].axvline(valid['n_frames'].median(), color='red', linestyle='--', label=f"Mediana={valid['n_frames'].median():.0f}")
axes[0].legend()

# 2) Distribuição de classes (emoções)
emotion_order = [EMOTION_MAP[i] for i in sorted(EMOTION_MAP.keys())]
emo_counts = valid['emotion_label'].value_counts().reindex(emotion_order, fill_value=0)
bars = axes[1].bar(range(len(emo_counts)), emo_counts.values, color='coral', edgecolor='white')
axes[1].set_xticks(range(len(emo_counts)))
axes[1].set_xticklabels(emo_counts.index, rotation=45, ha='right')
axes[1].set_ylabel('Contagem')
axes[1].set_title('Distribuição de Classes (Emoções)')
# Adicionar valores nas barras
for bar, val in zip(bars, emo_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(val), ha='center', va='bottom', fontsize=9)

# 3) Distribuição por ator
actor_cts = valid['actor_id'].value_counts().sort_index()
axes[2].bar(actor_cts.index, actor_cts.values, color='seagreen', edgecolor='white')
axes[2].set_xlabel('Actor ID')
axes[2].set_ylabel('Contagem')
axes[2].set_title('Amostras por Ator')

plt.tight_layout()
plt.savefig(os.path.join(QC_DIR, 'qc_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva em 03_qc/qc_distributions.png")

## 1.8 Tabela Resumo (pronta para copiar no artigo)

In [ ]:
# Tabela cruzada: emoção x ator
cross = pd.crosstab(valid['emotion_label'], valid['actor_id'], margins=True)
print("Tabela cruzada: Emoção x Ator")
print("=" * 80)
cross

In [ ]:
# Resumo final
print("\n" + "=" * 60)
print("RESUMO DO DATASET (para artigo)")
print("=" * 60)
print(f"Total de amostras:       {len(valid)}")
print(f"Atores:                  {valid['actor_id'].nunique()}")
print(f"Classes de emoção:       {valid['emotion_code'].nunique()}")
print(f"Features por frame:      {valid['n_features'].mode().iloc[0] if len(valid) > 0 else 'N/A'}")
print(f"Frames por amostra:      {valid['n_frames'].min()} - {valid['n_frames'].max()} (média={valid['n_frames'].mean():.1f})")
print(f"Amostras com NaN:        {valid['has_nan'].sum()}")
print("=" * 60)
print("\nNotebook 01 concluído com sucesso!")